# Independent Replication-10 – DAPT × Active Learning under Distribution Shift

Dieses Notebook führt eine **vorab festgelegte unabhängige Replikation** des explorativen DAPT×Active-Learning-Interaktionstests durch.

**Fixierte neue Pipeline-Seeds vor Ergebnisansicht:** `302, 312, 322, 332, 342, 352, 362, 372, 382, 392`.

Es werden **keine Seeds aufgrund ihrer Performance ausgeschlossen oder nachträglich ergänzt**.

## Primärer Test

\[
\Delta_{Int}=(DAPT_U-DAPT_R)-(T0_U-T0_R)
\]

bei **25 % Labels**, **0,5 % Ziel-FPR** und vier Stressszenarien: Temporal, Domain OOD, Template OOD, Domain+Template OOD.

## Protokolltreue

- frisches DAPT/MLM je Seed auf denselben 40k ungelabelten Webseiten
- DAPT: 1 Epoche, L256, MLM 0.15, Micro-Batch 4, GradAccum 8, LR 5e-5, WD 0.01, Warmup 10 %
- E2E: 5 Epochen, LR 2e-5, WD 0.01, Warmup 10 %, effektive Batchgröße 16
- gemeinsamer balancierter 10-%-Start pro Seed
- Random-Auswahl ist für T0 und DAPT identisch
- Uncertainty-Auswahl ist modellabhängig
- Labels werden erst **nach** der Auswahl als Reviewer-/Oracle-Labels verwendet

## Laufzeitsicherheit

Scores werden nach jedem Zustand gespeichert. Fertige Seeds werden über `seed_results/` und `audit/seed_*_COMPLETE.json` erkannt und bei einem Resume nicht neu gerechnet. Der große DAPT-Encoder wird nach vollständig abgeschlossenem Seed gelöscht, weil alle benötigten Scores persistiert sind.

In [ ]:
# ============================================================
# 00 – Imports, Konfiguration, präregistrierter Seed-Plan
# ============================================================
import os, gc, json, math, pickle, random, shutil, time, warnings, hashlib, zipfile
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForMaskedLM, AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling, get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix, precision_score, recall_score, f1_score
from scipy.stats import t as student_t

warnings.filterwarnings('ignore')
INPUT_ROOT=Path('/kaggle/input')
ROOT=Path('/kaggle/working/phreshphish_AL_SSL_INTERACTION_REPLICATION10')
DAPT_ROOT=ROOT/'dapt_encoders_partial'; STATE_ROOT=ROOT/'state_scores'; SEED_RESULT_ROOT=ROOT/'seed_results'
AUDIT_ROOT=ROOT/'audit'; TABLE_ROOT=ROOT/'tables'; FIG_ROOT=ROOT/'figures'; TOKEN_ROOT=ROOT/'tokens'
for p in [ROOT,DAPT_ROOT,STATE_ROOT,SEED_RESULT_ROOT,AUDIT_ROOT,TABLE_ROOT,FIG_ROOT,TOKEN_ROOT]: p.mkdir(parents=True,exist_ok=True)

REPLICATION10=[302,312,322,332,342,352,362,372,382,392]
RUN_SEEDS=list(REPLICATION10)
BUDGETS=[0.10,0.15,0.20,0.25]; AL_STEP=200
TARGET_FPRS=[0.005,0.010,0.020]; PRIMARY_FPR=0.005; PRIMARY_BUDGET=0.25
MAX_LENGTH=256; CALIBRATION_FRACTION=0.30; CALIBRATION_SPLIT_SEED=20260808; AP_BALANCE_SEED=20260809
DAPT_EPOCHS=1; DAPT_MICRO_BATCH=4; DAPT_GRAD_ACCUM=8; DAPT_LR=5e-5; DAPT_WEIGHT_DECAY=0.01; DAPT_WARMUP_RATIO=0.10; DAPT_MLM_PROB=0.15
E2E_EPOCHS=5; E2E_LR=2e-5; E2E_WEIGHT_DECAY=0.01; E2E_WARMUP_RATIO=0.10; E2E_BATCH_CANDIDATES=[16,8,4]; E2E_SCORE_BATCH=64
EXPECTED_STRESS_ROWS={'TEMPORAL':8000,'DOMAIN_OOD':3858,'TEMPLATE_OOD':6832,'DOMAIN_TEMPLATE_OOD':3708}
FINAL_STRESS=['TEMPORAL','DOMAIN_OOD','TEMPLATE_OOD','DOMAIN_TEMPLATE_OOD']
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available(): torch.backends.cuda.matmul.allow_tf32=False
seed_plan={'run':'INDEPENDENT_REPLICATION10_DAPT_X_ACTIVE_LEARNING','replication_seeds':REPLICATION10,'performance_based_seed_exclusions':[],'posthoc_seed_additions_allowed':False,'primary_metric':'recall','primary_budget':PRIMARY_BUDGET,'primary_target_fpr':PRIMARY_FPR,'primary_scenarios':FINAL_STRESS,'interaction':'(DAPT_U-DAPT_R)-(T0_U-T0_R)'}
(AUDIT_ROOT/'PREDECLARED_SEED_PLAN.json').write_text(json.dumps(seed_plan,indent=2),encoding='utf-8')
print({'run':seed_plan['run'],'torch':torch.__version__,'cuda':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'seeds':REPLICATION10,'output':str(ROOT)})

In [ ]:
# ============================================================
# 01 – Resume + Input Discovery
# ============================================================
def looks_like_rep(p):
    p=Path(p)
    return p.is_dir() and ((p/'REPLICATION10_PROGRESS.json').exists() or (p/'REPLICATION10_COMPLETE.json').exists() or (p/'seed_results').exists())
resume=[p for p in INPUT_ROOT.rglob('*') if looks_like_rep(p)]
RESUME_SOURCE=None
if resume:
    src=sorted(resume,key=lambda p:len(str(p)))[0]
    for item in src.iterdir():
        dst=ROOT/item.name
        if item.is_dir(): shutil.copytree(item,dst,dirs_exist_ok=True)
        elif not dst.exists(): shutil.copy2(item,dst)
    RESUME_SOURCE=str(src)
print({'resume':RESUME_SOURCE})

def has_hf_model(p):
    p=Path(p); return p.is_dir() and (p/'config.json').exists() and ((p/'model.safetensors').exists() or (p/'pytorch_model.bin').exists())
split_hits=list(INPUT_ROOT.rglob('split_roles_and_holdout_cache_v2_ram_safe.pkl'))
bundle_hits=list(INPUT_ROOT.rglob('dapt40k_bundle.pkl'))
if not split_hits or not bundle_hits: raise FileNotFoundError('STOP: ENDGAME Split-Cache oder dapt40k_bundle.pkl fehlt.')
SPLIT_PATH=split_hits[0]; DAPT_BUNDLE_PATH=bundle_hits[0]
base=[p for p in INPUT_ROOT.rglob('roberta-base') if has_hf_model(p)]
if not base: raise FileNotFoundError('STOP: roberta-base fehlt.')
BASE_MODEL_DIR=sorted(base,key=lambda p:len(str(p)))[0]

def token_root_ok(p):
    return all((p/n/'input_ids.npy').exists() and (p/n/'attention_mask.npy').exists() for n in ['train_4k','calibration','iid_test','holdout_8k'])
token_roots=[p for p in INPUT_ROOT.rglob('tokens') if token_root_ok(p)]
if not token_roots: raise FileNotFoundError('STOP: TokenCache/tokens fehlt.')
SOURCE_TOKEN_ROOT=sorted(token_roots,key=lambda p:len(str(p)))[0]
print(json.dumps({'split':str(SPLIT_PATH),'dapt_bundle':str(DAPT_BUNDLE_PATH),'base_model':str(BASE_MODEL_DIR),'tokens':str(SOURCE_TOKEN_ROOT)},indent=2))

In [ ]:
# ============================================================
# 02 – Datenrollen, Stressszenarien, Token-Caches
# ============================================================
def load_pickle(path):
    with open(path,'rb') as f: return pickle.load(f)
def first_existing(obj,keys):
    if not isinstance(obj,dict): return None
    for k in keys:
        if k in obj: return obj[k]
    return None
split_payload=load_pickle(SPLIT_PATH); dapt_payload=load_pickle(DAPT_BUNDLE_PATH)
train_df=first_existing(split_payload,['train_df','train','supervised_train','downstream_train'])
val_df=first_existing(split_payload,['val_df','validation_df','validation','val'])
holdout_df=first_existing(split_payload,['final_holdout_clean','final_holdout','holdout_df','holdout'])
pretrain_df=first_existing(dapt_payload,['pretrain_large_df','frame','pretrain_df','pretrain'])
dapt_holdout=first_existing(dapt_payload,['final_holdout_clean'])
for name,obj in [('train',train_df),('validation',val_df),('holdout',holdout_df),('pretrain40k',pretrain_df)]:
    if not isinstance(obj,pd.DataFrame): raise TypeError(f'STOP: {name} kein DataFrame')
train_df=train_df.reset_index(drop=True).copy(); val_df=val_df.reset_index(drop=True).copy(); holdout_df=holdout_df.reset_index(drop=True).copy(); pretrain_df=pretrain_df.reset_index(drop=True).copy()
if len(train_df)!=4000 or len(pretrain_df)!=40000: raise RuntimeError(f'STOP: train={len(train_df)}, pretrain={len(pretrain_df)}')
if isinstance(dapt_holdout,pd.DataFrame):
    dapt_holdout=dapt_holdout.reset_index(drop=True); lookup=dapt_holdout.copy(); lookup.index=lookup['sha256'].astype(str); hs=holdout_df['sha256'].astype(str)
    for c in ['near_duplicate_to_development','min_simhash_distance_to_development','template_seen_in_development']:
        if c in lookup.columns:
            mapped=hs.map(lookup[c])
            if mapped.isna().any(): raise RuntimeError(f'STOP: Holdout flag {c} unvollständig')
            holdout_df[c]=mapped.to_numpy()
cal_idx,iid_idx=train_test_split(np.arange(len(val_df)),test_size=1.0-CALIBRATION_FRACTION,random_state=CALIBRATION_SPLIT_SEED,stratify=val_df['label'].to_numpy())
calibration_df=val_df.iloc[np.sort(cal_idx)].reset_index(drop=True); iid_df=val_df.iloc[np.sort(iid_idx)].reset_index(drop=True)

dev_domains=set(); dev_templates=set()
for frame in [pretrain_df,train_df,val_df]:
    dev_domains.update(frame['domain'].fillna('').astype(str).tolist()); dev_templates.update(frame['template_hash'].fillna('').astype(str).tolist())
dev_domains.discard(''); dev_templates.discard('')
h_domain=holdout_df['domain'].fillna('').astype(str); h_template=holdout_df['template_hash'].fillna('').astype(str)
domain_seen=h_domain.isin(dev_domains).to_numpy(); template_seen=h_template.isin(dev_templates).to_numpy()
near_dup=holdout_df['near_duplicate_to_development'].fillna(False).astype(bool).to_numpy()
domain_new=(~domain_seen)&h_domain.ne('').to_numpy(); template_ood=(~template_seen)&(~near_dup)&h_template.ne('').to_numpy(); domain_template=domain_new&template_ood

def balanced_indices(frame,mask,seed):
    sub=frame.loc[np.asarray(mask)].copy(); n=min(int((sub.label==0).sum()),int((sub.label==1).sum()))
    p0=sub[sub.label.eq(0)].sample(n=n,random_state=seed); p1=sub[sub.label.eq(1)].sample(n=n,random_state=seed+1)
    return np.sort(pd.concat([p0,p1]).index.to_numpy(dtype=np.int32))
stress={'TEMPORAL':np.arange(len(holdout_df),dtype=np.int32),'DOMAIN_OOD':balanced_indices(holdout_df,domain_new,108),'TEMPLATE_OOD':balanced_indices(holdout_df,template_ood,109),'DOMAIN_TEMPLATE_OOD':balanced_indices(holdout_df,domain_template,110)}
for k,v in EXPECTED_STRESS_ROWS.items():
    if len(stress[k])!=v: raise RuntimeError(f'STOP: {k} {len(stress[k])}!={v}')
iid_y=iid_df['label'].to_numpy(dtype=int); pos=np.flatnonzero(iid_y==1); neg=np.flatnonzero(iid_y==0); n=min(len(pos),len(neg)); rng=np.random.default_rng(AP_BALANCE_SEED)
IID_BALANCED_IDX=np.sort(np.concatenate([rng.choice(pos,n,replace=False),rng.choice(neg,n,replace=False)]).astype(np.int32))
SCENARIOS={'IID_ORIGINAL':('iid',np.arange(len(iid_df),dtype=np.int32)),'IID_BALANCED':('iid',IID_BALANCED_IDX),'TEMPORAL':('holdout',stress['TEMPORAL']),'DOMAIN_OOD':('holdout',stress['DOMAIN_OOD']),'TEMPLATE_OOD':('holdout',stress['TEMPLATE_OOD']),'DOMAIN_TEMPLATE_OOD':('holdout',stress['DOMAIN_TEMPLATE_OOD'])}
y_train=train_df['label'].to_numpy(dtype=int); ycal=calibration_df['label'].to_numpy(dtype=int)

def load_token_dir(root): return {'input_ids':np.load(root/'input_ids.npy',mmap_mode='r'),'attention_mask':np.load(root/'attention_mask.npy',mmap_mode='r')}
TOKENS={'train':load_token_dir(SOURCE_TOKEN_ROOT/'train_4k'),'calibration':load_token_dir(SOURCE_TOKEN_ROOT/'calibration'),'iid':load_token_dir(SOURCE_TOKEN_ROOT/'iid_test'),'holdout':load_token_dir(SOURCE_TOKEN_ROOT/'holdout_8k')}
expected={'train':len(train_df),'calibration':len(calibration_df),'iid':len(iid_df),'holdout':len(holdout_df)}
for s,a in TOKENS.items():
    if a['input_ids'].shape!=(expected[s],MAX_LENGTH): raise RuntimeError(f'STOP token shape {s}: {a["input_ids"].shape}')

tokenizer=AutoTokenizer.from_pretrained(str(BASE_MODEL_DIR),local_files_only=True,use_fast=True)
PRETRAIN_TOKENS=None; PRETRAIN_TOKEN_SOURCE=None
for name in ['pretrain_40k','pretrain_dapt_40k']:
    p=SOURCE_TOKEN_ROOT/name
    if (p/'input_ids.npy').exists() and (p/'attention_mask.npy').exists():
        a=load_token_dir(p)
        if a['input_ids'].shape==(len(pretrain_df),MAX_LENGTH): PRETRAIN_TOKENS=a; PRETRAIN_TOKEN_SOURCE=str(p); break
if PRETRAIN_TOKENS is None:
    p=TOKEN_ROOT/'pretrain_40k'; p.mkdir(parents=True,exist_ok=True); ids_path=p/'input_ids.npy'; mask_path=p/'attention_mask.npy'
    if not (ids_path.exists() and mask_path.exists()):
        N=len(pretrain_df); ids=np.lib.format.open_memmap(ids_path,mode='w+',dtype=np.int32,shape=(N,MAX_LENGTH)); masks=np.lib.format.open_memmap(mask_path,mode='w+',dtype=np.uint8,shape=(N,MAX_LENGTH)); texts=pretrain_df['text'].fillna('').astype(str).tolist()
        for st in range(0,N,512):
            en=min(st+512,N); enc=tokenizer(texts[st:en],padding='max_length',truncation=True,max_length=MAX_LENGTH,return_tensors='np'); ids[st:en]=enc['input_ids'].astype(np.int32); masks[st:en]=enc['attention_mask'].astype(np.uint8)
            if en%5000==0 or en==N: print({'tokenizing40k':en,'total':N})
        ids.flush(); masks.flush(); del ids,masks,texts; gc.collect()
    PRETRAIN_TOKENS=load_token_dir(p); PRETRAIN_TOKEN_SOURCE=str(p)
print({'data':'PASS','train':len(train_df),'pretrain':len(pretrain_df),'stress':{k:len(v) for k,v in stress.items()},'pretrain_tokens':PRETRAIN_TOKEN_SOURCE})

In [ ]:
# ============================================================
# 03 – Helper: Seeds, Metriken, Datasets, Labelpfade
# ============================================================
def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
def labeled_hash(idx):
    a=np.sort(np.asarray(idx,dtype=np.int32)); return hashlib.sha256(a.tobytes()).hexdigest()
def write_progress(stage,**kw):
    (ROOT/'REPLICATION10_PROGRESS.json').write_text(json.dumps({'stage':stage,'utc':pd.Timestamp.utcnow().isoformat(),'seeds':REPLICATION10,**kw},indent=2),encoding='utf-8')
def threshold_for_target_fpr(y,score,target):
    y=np.asarray(y,dtype=int); s=np.asarray(score,dtype=float); neg=np.sort(s[y==0])[::-1]; allowed=int(math.floor(target*len(neg)+1e-12))
    if allowed<=0: return float(np.nextafter(neg[0],np.inf))
    if allowed>=len(neg): return float(-np.inf)
    return float(np.nextafter(neg[allowed],np.inf))
def metric_row(y,score,thr):
    y=np.asarray(y,dtype=int); score=np.asarray(score,dtype=float); pred=(score>=thr).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'average_precision':float(average_precision_score(y,score)),'roc_auc':float(roc_auc_score(y,score)) if len(np.unique(y))==2 else np.nan,'precision':float(precision_score(y,pred,zero_division=0)),'recall':float(recall_score(y,pred,zero_division=0)),'f1':float(f1_score(y,pred,zero_division=0)),'empirical_fpr':float(fp/max(fp+tn,1)),'tp':int(tp),'fp':int(fp),'tn':int(tn),'fn':int(fn)}
def calibration_ops(score):
    out={}
    for t in TARGET_FPRS:
        thr=threshold_for_target_fpr(ycal,score,t); m=metric_row(ycal,score,thr); out[t]={'threshold':thr,'calibration_empirical_fpr':m['empirical_fpr']}
    return out
def scenario_view(iid_score,holdout_score,scenario):
    src,idx=SCENARIOS[scenario]
    if src=='iid': return iid_df.iloc[idx]['label'].to_numpy(dtype=int),np.asarray(iid_score)[idx]
    return holdout_df.iloc[idx]['label'].to_numpy(dtype=int),np.asarray(holdout_score)[idx]

def initial_10pct(seed):
    rng=np.random.default_rng(seed); parts=[]
    for c in [0,1]:
        idx=np.flatnonzero(y_train==c).copy(); rng.shuffle(idx); parts.append(idx[:max(1,int(round(len(idx)*.10)))])
    out=np.sort(np.concatenate(parts)).astype(np.int32)
    if len(out)!=400 or int((y_train[out]==0).sum())!=200 or int((y_train[out]==1).sum())!=200: raise RuntimeError(f'initial invalid {seed}')
    return out
def random_trajectory(seed):
    labeled=initial_10pct(seed); hist={.10:labeled.copy()}
    for round_i,next_budget in enumerate([.15,.20,.25]):
        unl=np.setdiff1d(np.arange(len(y_train),dtype=np.int32),labeled); rng=np.random.default_rng(seed+70000+1000*round_i); chosen=rng.choice(unl,size=AL_STEP,replace=False); labeled=np.sort(np.concatenate([labeled,chosen])).astype(np.int32); hist[next_budget]=labeled.copy()
    return hist

class MLMArrayDataset(Dataset):
    def __init__(self,a): self.ids=a['input_ids']; self.mask=a['attention_mask']
    def __len__(self): return len(self.ids)
    def __getitem__(self,i): return {'input_ids':torch.from_numpy(np.asarray(self.ids[i],dtype=np.int64)),'attention_mask':torch.from_numpy(np.asarray(self.mask[i],dtype=np.int64))}
class TokenSubsetDataset(Dataset):
    def __init__(self,a,y,idx): self.ids=a['input_ids']; self.mask=a['attention_mask']; self.y=np.asarray(y,dtype=np.int64); self.idx=np.asarray(idx,dtype=np.int64)
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        j=int(self.idx[i]); return torch.tensor(self.ids[j],dtype=torch.long),torch.tensor(self.mask[j],dtype=torch.long),torch.tensor(self.y[j],dtype=torch.long)
class TokenAllDataset(Dataset):
    def __init__(self,a): self.ids=a['input_ids']; self.mask=a['attention_mask']
    def __len__(self): return len(self.ids)
    def __getitem__(self,i): return torch.tensor(self.ids[i],dtype=torch.long),torch.tensor(self.mask[i],dtype=torch.long)
@torch.no_grad()
def score_model(model,a,batch_size=E2E_SCORE_BATCH):
    ds=TokenAllDataset(a); loader=DataLoader(ds,batch_size=batch_size,shuffle=False,num_workers=2,pin_memory=torch.cuda.is_available()); model.eval(); out=[]
    for ids,mask in loader:
        ids=ids.to(DEVICE,non_blocking=True); mask=mask.to(DEVICE,non_blocking=True)
        with torch.amp.autocast('cuda',dtype=torch.float16,enabled=torch.cuda.is_available()): logits=model(input_ids=ids,attention_mask=mask).logits
        out.append(torch.softmax(logits.float(),dim=-1)[:,1].cpu().numpy())
    return np.concatenate(out).astype(np.float32)

In [ ]:
# ============================================================
# 04 – Exakt FINAL-N10: frisches DAPT/MLM pro Seed
# ============================================================
mlm_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=True,mlm_probability=DAPT_MLM_PROB)
def hf_dir_ok(p): return p.is_dir() and (p/'config.json').exists() and ((p/'model.safetensors').exists() or (p/'pytorch_model.bin').exists())
def dapt_dir(seed): return DAPT_ROOT/f'seed_{seed}'
def train_or_reuse_dapt(seed):
    out=dapt_dir(seed)
    if hf_dir_ok(out): print({'DAPT':seed,'status':'REUSE'}); return out
    set_all_seeds(seed); model=AutoModelForMaskedLM.from_pretrained(str(BASE_MODEL_DIR),local_files_only=True).to(DEVICE); ds=MLMArrayDataset(PRETRAIN_TOKENS); g=torch.Generator(); g.manual_seed(seed)
    loader=DataLoader(ds,batch_size=DAPT_MICRO_BATCH,shuffle=True,generator=g,num_workers=2,pin_memory=torch.cuda.is_available(),collate_fn=mlm_collator)
    updates_per_epoch=math.ceil(len(loader)/DAPT_GRAD_ACCUM); total_updates=updates_per_epoch*DAPT_EPOCHS
    opt=torch.optim.AdamW(model.parameters(),lr=DAPT_LR,weight_decay=DAPT_WEIGHT_DECAY); sched=get_linear_schedule_with_warmup(opt,num_warmup_steps=int(round(total_updates*DAPT_WARMUP_RATIO)),num_training_steps=total_updates); scaler=torch.amp.GradScaler('cuda',enabled=torch.cuda.is_available())
    model.train(); opt.zero_grad(set_to_none=True); running=0.; micro=0; update=0; hist=[]; started=time.perf_counter()
    for epoch in range(DAPT_EPOCHS):
        for step,batch in enumerate(loader):
            ids=batch['input_ids'].to(DEVICE,non_blocking=True); mask=batch['attention_mask'].to(DEVICE,non_blocking=True); labels=batch['labels'].to(DEVICE,non_blocking=True)
            with torch.amp.autocast('cuda',dtype=torch.float16,enabled=torch.cuda.is_available()): loss=model(input_ids=ids,attention_mask=mask,labels=labels).loss/DAPT_GRAD_ACCUM
            scaler.scale(loss).backward(); running+=float(loss.detach().cpu())*DAPT_GRAD_ACCUM; micro+=1
            if (step+1)%DAPT_GRAD_ACCUM==0 or step+1==len(loader):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); update+=1
                if update%250==0 or update==total_updates:
                    row={'seed':seed,'update':update,'total_updates':total_updates,'mean_loss':running/max(micro,1),'seconds':time.perf_counter()-started}; hist.append(row); print(row)
    out.mkdir(parents=True,exist_ok=True); model.roberta.save_pretrained(out); tokenizer.save_pretrained(out)
    (out/'dapt_config.json').write_text(json.dumps({'seed':seed,'objective':'MLM','rows':len(pretrain_df),'epochs':DAPT_EPOCHS,'micro_batch':DAPT_MICRO_BATCH,'gradient_accumulation':DAPT_GRAD_ACCUM,'effective_batch':DAPT_MICRO_BATCH*DAPT_GRAD_ACCUM,'learning_rate':DAPT_LR,'weight_decay':DAPT_WEIGHT_DECAY,'warmup_ratio':DAPT_WARMUP_RATIO,'mlm_probability':DAPT_MLM_PROB,'max_length':MAX_LENGTH,'labels_used':False,'evaluation_data_used':False},indent=2),encoding='utf-8')
    pd.DataFrame(hist).to_csv(AUDIT_ROOT/f'dapt_history_seed{seed}.csv',index=False)
    del model,opt,sched,scaler,loader,ds; gc.collect();
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    write_progress('DAPT_COMPLETE',seed=seed); return out

In [ ]:
# ============================================================
# 05 – E2E-Fine-Tuning, Score-State, Active Learning
# ============================================================
def state_path(model_name,strategy,seed,budget): return STATE_ROOT/f'{model_name}_{strategy}_seed{seed}_budget{str(float(budget)).replace(".","p")}.npz'
def fit_e2e(init_dir,seed,round_i,labeled_idx):
    train_seed=seed+1000*round_i; last=None
    for bs in E2E_BATCH_CANDIDATES:
        try:
            set_all_seeds(train_seed); model=AutoModelForSequenceClassification.from_pretrained(str(init_dir),num_labels=2,local_files_only=True,ignore_mismatched_sizes=True).to(DEVICE); ds=TokenSubsetDataset(TOKENS['train'],y_train,labeled_idx); g=torch.Generator(); g.manual_seed(train_seed)
            loader=DataLoader(ds,batch_size=bs,shuffle=True,generator=g,num_workers=2,pin_memory=torch.cuda.is_available()); ga=max(1,16//bs); total_updates=math.ceil(len(loader)/ga)*E2E_EPOCHS
            opt=torch.optim.AdamW(model.parameters(),lr=E2E_LR,weight_decay=E2E_WEIGHT_DECAY); sched=get_linear_schedule_with_warmup(opt,num_warmup_steps=int(round(total_updates*E2E_WARMUP_RATIO)),num_training_steps=total_updates); scaler=torch.amp.GradScaler('cuda',enabled=torch.cuda.is_available()); started=time.perf_counter()
            for ep in range(E2E_EPOCHS):
                model.train(); opt.zero_grad(set_to_none=True); losses=[]
                for step,(ids,mask,labels) in enumerate(loader):
                    ids=ids.to(DEVICE,non_blocking=True); mask=mask.to(DEVICE,non_blocking=True); labels=labels.to(DEVICE,non_blocking=True)
                    with torch.amp.autocast('cuda',dtype=torch.float16,enabled=torch.cuda.is_available()): loss=model(input_ids=ids,attention_mask=mask,labels=labels).loss/ga
                    scaler.scale(loss).backward(); losses.append(float(loss.detach().cpu())*ga)
                    if (step+1)%ga==0 or step+1==len(loader):
                        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
                print({'E2E_seed':seed,'round':round_i,'epoch':ep+1,'n_labeled':len(labeled_idx),'loss':float(np.mean(losses))})
            return model,time.perf_counter()-started,bs,ga
        except RuntimeError as e:
            last=e
            if 'out of memory' not in str(e).lower(): raise
            try: del model
            except Exception: pass
            gc.collect();
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    raise RuntimeError(f'E2E failed seed={seed}, round={round_i}') from last

def build_or_load_state(model_name,strategy,seed,budget,round_i,labeled_idx,init_dir,full_eval):
    cache_strategy='SHARED_INITIAL' if budget==.10 else strategy; path=state_path(model_name,cache_strategy,seed,budget); expected=labeled_hash(labeled_idx)
    if path.exists():
        d=np.load(path,allow_pickle=False); ok=('labeled_hash' in d.files and str(d['labeled_hash'][0])==expected); eval_ok=(not full_eval or all(k in d.files for k in ['cal_score','iid_score','holdout_score']))
        if ok and eval_ok: print({'STATE_REUSE':True,'model':model_name,'strategy':cache_strategy,'seed':seed,'budget':budget}); return {k:np.asarray(d[k]) for k in d.files}
    model,fit,bs,ga=fit_e2e(init_dir,seed,round_i,labeled_idx); payload={'labeled_hash':np.asarray([expected]),'n_labeled':np.asarray([len(labeled_idx)],dtype=np.int32),'n_benign':np.asarray([int((y_train[labeled_idx]==0).sum())],dtype=np.int32),'n_phish':np.asarray([int((y_train[labeled_idx]==1).sum())],dtype=np.int32),'train_score':score_model(model,TOKENS['train']),'fit_seconds':np.asarray([fit]),'batch_size':np.asarray([bs]),'grad_accum':np.asarray([ga])}
    if full_eval: payload.update({'cal_score':score_model(model,TOKENS['calibration']),'iid_score':score_model(model,TOKENS['iid']),'holdout_score':score_model(model,TOKENS['holdout'])})
    np.savez_compressed(path,**payload); del model; gc.collect();
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    write_progress('STATE_COMPLETE',model=model_name,strategy=cache_strategy,seed=seed,budget=budget); return payload

def uncertainty_select(score,unlabeled):
    unlabeled=np.asarray(unlabeled,dtype=np.int32); u=np.abs(np.asarray(score,dtype=float)[unlabeled]-.5); order=np.argsort(u); return np.sort(unlabeled[order[:AL_STEP]].astype(np.int32)),u[order[:AL_STEP]]
def run_uncertainty(model_name,seed,init_dir):
    labeled=initial_10pct(seed); selection={'0.10':labeled.tolist()}; acquisitions=[]
    for round_i,budget in enumerate(BUDGETS):
        if len(labeled)!=int(round(len(y_train)*budget)): raise RuntimeError(f'budget mismatch {model_name}/{seed}/{budget}')
        full=budget==.25; state=build_or_load_state(model_name,'UNCERTAINTY_GLOBAL',seed,budget,round_i,labeled,init_dir,full); selection[f'{budget:.2f}']=labeled.tolist()
        if full: return state,selection,acquisitions
        unl=np.setdiff1d(np.arange(len(y_train),dtype=np.int32),labeled); chosen,dist=uncertainty_select(state['train_score'],unl); revealed=y_train[chosen]
        acquisitions.append({'model':model_name,'strategy':'UNCERTAINTY_GLOBAL','seed':seed,'from_budget':budget,'to_budget':BUDGETS[round_i+1],'selected_n':len(chosen),'selected_benign':int((revealed==0).sum()),'selected_phish':int((revealed==1).sum()),'selected_phish_share':float((revealed==1).mean()),'mean_abs_distance_to_0p5':float(np.mean(dist))})
        labeled=np.sort(np.concatenate([labeled,chosen])).astype(np.int32)
    raise RuntimeError('trajectory failed')

def evaluate_state(model_name,strategy,seed,state):
    rows=[]; ops=calibration_ops(state['cal_score'])
    for target in TARGET_FPRS:
        op=ops[target]
        for scenario in SCENARIOS:
            y,score=scenario_view(state['iid_score'],state['holdout_score'],scenario); m=metric_row(y,score,op['threshold'])
            rows.append({'model':model_name,'strategy':strategy,'seed':seed,'label_budget':.25,'n_labeled':int(state['n_labeled'][0]),'labeled_benign':int(state['n_benign'][0]),'labeled_phish':int(state['n_phish'][0]),'scenario':scenario,'target_fpr':target,'threshold':op['threshold'],'calibration_empirical_fpr':op['calibration_empirical_fpr'],'recall':m['recall'],'empirical_fpr':m['empirical_fpr'],'precision':m['precision'],'f1':m['f1'],'average_precision':m['average_precision'],'roc_auc':m['roc_auc'],'tp':m['tp'],'fp':m['fp'],'tn':m['tn'],'fn':m['fn'],'fit_seconds':float(state['fit_seconds'][0])})
    return rows

In [ ]:
# ============================================================
# 06 – Hauptlauf: 10 frische Pipeline-Seeds
# ============================================================
for seed in RUN_SEEDS:
    result_path=SEED_RESULT_ROOT/f'seed_{seed}_results.csv'; complete_path=AUDIT_ROOT/f'seed_{seed}_COMPLETE.json'
    if result_path.exists() and complete_path.exists(): print({'SEED':seed,'status':'REUSE_COMPLETE'}); continue
    print('\n'+'='*80); print('INDEPENDENT REPLICATION SEED',seed); print('='*80)
    dapt_init=train_or_reuse_dapt(seed); random_hist=random_trajectory(seed); random25=random_hist[.25]
    t0_u,t0_sel,t0_acq=run_uncertainty('T0_E2E',seed,BASE_MODEL_DIR); dapt_u,dapt_sel,dapt_acq=run_uncertainty('DAPT_E2E',seed,dapt_init)
    t0_r=build_or_load_state('T0_E2E','RANDOM_GLOBAL',seed,.25,3,random25,BASE_MODEL_DIR,True); dapt_r=build_or_load_state('DAPT_E2E','RANDOM_GLOBAL',seed,.25,3,random25,dapt_init,True)
    rows=[]; rows+=evaluate_state('T0_E2E','UNCERTAINTY_GLOBAL',seed,t0_u); rows+=evaluate_state('T0_E2E','RANDOM_GLOBAL',seed,t0_r); rows+=evaluate_state('DAPT_E2E','UNCERTAINTY_GLOBAL',seed,dapt_u); rows+=evaluate_state('DAPT_E2E','RANDOM_GLOBAL',seed,dapt_r)
    sdf=pd.DataFrame(rows); expected=4*len(SCENARIOS)*len(TARGET_FPRS)
    if len(sdf)!=expected: raise RuntimeError(f'seed rows {len(sdf)}!={expected}')
    sdf.to_csv(result_path,index=False)
    selection={'seed':seed,'initial_10pct':initial_10pct(seed).tolist(),'random':{f'{b:.2f}':v.tolist() for b,v in random_hist.items()},'t0_uncertainty':t0_sel,'dapt_uncertainty':dapt_sel,'random_T0_DAPT_identical':True}
    (AUDIT_ROOT/f'seed_{seed}_selection.json').write_text(json.dumps(selection),encoding='utf-8'); pd.DataFrame(t0_acq+dapt_acq).to_csv(AUDIT_ROOT/f'seed_{seed}_acquisition.csv',index=False)
    complete={'seed':seed,'status':'COMPLETE','fresh_dapt':True,'random_shared_between_models':True,'initial_shared':True,'n_result_rows':len(sdf),'t0_uncertainty_25_hash':labeled_hash(np.asarray(t0_sel['0.25'],dtype=np.int32)),'dapt_uncertainty_25_hash':labeled_hash(np.asarray(dapt_sel['0.25'],dtype=np.int32)),'random_25_hash':labeled_hash(random25),'performance_based_exclusion':False}
    complete_path.write_text(json.dumps(complete,indent=2),encoding='utf-8')
    if dapt_init.exists(): shutil.rmtree(dapt_init,ignore_errors=True)
    del t0_u,dapt_u,t0_r,dapt_r; gc.collect();
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    completed=[s for s in REPLICATION10 if (AUDIT_ROOT/f'seed_{s}_COMPLETE.json').exists() and (SEED_RESULT_ROOT/f'seed_{s}_results.csv').exists()]
    write_progress('SEED_COMPLETE',seed=seed,completed_seeds=completed,n_completed=len(completed))
print({'completed':[s for s in REPLICATION10 if (AUDIT_ROOT/f'seed_{s}_COMPLETE.json').exists()]})

In [ ]:
# ============================================================
# 07 – Aggregation + primäre Statistik
# ============================================================
completed=[s for s in REPLICATION10 if (AUDIT_ROOT/f'seed_{s}_COMPLETE.json').exists() and (SEED_RESULT_ROOT/f'seed_{s}_results.csv').exists()]
if set(RUN_SEEDS)==set(REPLICATION10) and completed!=REPLICATION10: raise RuntimeError(f'STOP: nur {completed} komplett')
RESULTS=pd.concat([pd.read_csv(SEED_RESULT_ROOT/f'seed_{s}_results.csv') for s in completed],ignore_index=True)
expected=len(completed)*4*len(SCENARIOS)*len(TARGET_FPRS)
if len(RESULTS)!=expected: raise RuntimeError(f'rows {len(RESULTS)}!={expected}')
RESULTS.to_csv(ROOT/'REPLICATION10_results_long.csv',index=False)
acqs=[pd.read_csv(AUDIT_ROOT/f'seed_{s}_acquisition.csv') for s in completed if (AUDIT_ROOT/f'seed_{s}_acquisition.csv').exists()]
if acqs: pd.concat(acqs,ignore_index=True).to_csv(ROOT/'REPLICATION10_acquisition_audit.csv',index=False)

def exact_signflip(diff):
    d=np.asarray(diff,dtype=float); d=d[np.isfinite(d)]; obs=abs(d.mean()); vals=[abs(np.mean(d*np.asarray(signs))) for signs in product([-1.,1.],repeat=len(d))]; return float(np.mean(np.asarray(vals)>=obs-1e-15))
def paired_stats(diff):
    d=np.asarray(diff,dtype=float); d=d[np.isfinite(d)]; n=len(d); mean=float(d.mean()); sd=float(d.std(ddof=1)) if n>1 else np.nan; sem=sd/math.sqrt(n) if n>1 else np.nan; crit=float(student_t.ppf(.975,n-1)) if n>1 else np.nan
    return {'n_seeds':n,'mean_difference':mean,'sd_difference':sd,'ci95_low':mean-crit*sem if n>1 else np.nan,'ci95_high':mean+crit*sem if n>1 else np.nan,'exact_signflip_p_two_sided':exact_signflip(d),'cohen_dz':mean/sd if n>1 and sd>0 else np.nan,'wins':int((d>1e-12).sum()),'ties':int((np.abs(d)<=1e-12).sum()),'losses':int((d<-1e-12).sum())}
def holm_adjust(p):
    p=np.asarray(p,dtype=float); m=len(p); order=np.argsort(p); out=np.empty(m); running=0.
    for rank,idx in enumerate(order):
        val=min(1.,(m-rank)*p[idx]); running=max(running,val); out[idx]=running
    return out
METRICS=['recall','empirical_fpr','precision','f1','average_precision','roc_auc']; stat_rows=[]
for target in TARGET_FPRS:
    for scenario in SCENARIOS:
        for metric in METRICS:
            gains={}
            for model in ['T0_E2E','DAPT_E2E']:
                u=RESULTS[(RESULTS.model==model)&(RESULTS.strategy=='UNCERTAINTY_GLOBAL')&(RESULTS.scenario==scenario)&(RESULTS.target_fpr==target)].set_index('seed'); r=RESULTS[(RESULTS.model==model)&(RESULTS.strategy=='RANDOM_GLOBAL')&(RESULTS.scenario==scenario)&(RESULTS.target_fpr==target)].set_index('seed'); diff=u.loc[completed,metric].to_numpy()-r.loc[completed,metric].to_numpy(); gains[model]=diff
                stat_rows.append({'contrast':f'AL_GAIN_{model}_UNCERTAINTY_MINUS_RANDOM','model':model,'scenario':scenario,'target_fpr':target,'label_budget':.25,'metric':metric,**paired_stats(diff)})
            interaction=gains['DAPT_E2E']-gains['T0_E2E']; stat_rows.append({'contrast':'INTERACTION_(DAPT_U-DAPT_R)-(T0_U-T0_R)','model':'DAPT_vs_T0','scenario':scenario,'target_fpr':target,'label_budget':.25,'metric':metric,**paired_stats(interaction)})
STATS=pd.DataFrame(stat_rows); pm=STATS.contrast.str.startswith('INTERACTION_')&STATS.metric.eq('recall')&STATS.target_fpr.eq(PRIMARY_FPR)&STATS.scenario.isin(FINAL_STRESS); STATS['holm_p_primary4']=np.nan; idx=STATS.index[pm]
if len(idx)==4: STATS.loc[idx,'holm_p_primary4']=holm_adjust(STATS.loc[idx,'exact_signflip_p_two_sided'].to_numpy())
STATS.to_csv(ROOT/'REPLICATION10_statistics.csv',index=False); PRIMARY=STATS[pm].copy(); PRIMARY.to_csv(TABLE_ROOT/'TABLE01_primary_interaction_replication10.csv',index=False)
print(PRIMARY[['scenario','n_seeds','mean_difference','ci95_low','ci95_high','exact_signflip_p_two_sided','holm_p_primary4','wins','losses']].to_string(index=False))

In [ ]:
# ============================================================
# 08 – Stressmittel + Completion + Analyse-ZIP
# ============================================================
pr=RESULTS[RESULTS.scenario.isin(FINAL_STRESS)&RESULTS.target_fpr.eq(PRIMARY_FPR)].copy()
stress_seed=pr.groupby(['model','strategy','seed'],as_index=False).agg(recall=('recall','mean'),empirical_fpr=('empirical_fpr','mean'),average_precision=('average_precision','mean'),roc_auc=('roc_auc','mean'))
stress_summary=stress_seed.groupby(['model','strategy'],as_index=False).agg(recall_mean=('recall','mean'),recall_std=('recall','std'),empirical_fpr_mean=('empirical_fpr','mean'),ap_mean=('average_precision','mean'),auc_mean=('roc_auc','mean')); stress_summary.to_csv(TABLE_ROOT/'TABLE02_stress_mean_four_conditions.csv',index=False)
gains={}; rows=[]
for model in ['T0_E2E','DAPT_E2E']:
    u=stress_seed[(stress_seed.model==model)&(stress_seed.strategy=='UNCERTAINTY_GLOBAL')].set_index('seed'); r=stress_seed[(stress_seed.model==model)&(stress_seed.strategy=='RANDOM_GLOBAL')].set_index('seed'); gain=u.loc[completed,'recall'].to_numpy()-r.loc[completed,'recall'].to_numpy(); gains[model]=gain; rows.append({'contrast':f'AL_GAIN_{model}',**paired_stats(gain)})
interaction=gains['DAPT_E2E']-gains['T0_E2E']; rows.append({'contrast':'INTERACTION_DAPT_AL_GAIN_MINUS_T0_AL_GAIN',**paired_stats(interaction)})
STRESS_STATS=pd.DataFrame(rows); STRESS_STATS.to_csv(TABLE_ROOT/'TABLE03_stress_mean_interaction_replication10.csv',index=False)
print(stress_summary.to_string(index=False)); print(STRESS_STATS.to_string(index=False))

# Drei kompakte Figures.
plot=stress_summary.copy(); plot['condition']=plot['model'].str.replace('_E2E','',regex=False)+' | '+plot['strategy'].map({'RANDOM_GLOBAL':'Random','UNCERTAINTY_GLOBAL':'Uncertainty'})
fig,ax=plt.subplots(figsize=(8.5,5)); x=np.arange(len(plot)); ax.bar(x,100*plot.recall_mean); ax.set_xticks(x); ax.set_xticklabels(plot.condition,rotation=20,ha='right'); ax.set_ylabel('Mittlerer Recall über vier Stressszenarien [%]'); ax.set_title('Independent Replication10 – 25 % Labels, 0,5 % Ziel-FPR'); fig.tight_layout(); fig.savefig(FIG_ROOT/'FIG01_four_conditions.png',dpi=220); plt.close(fig)
p=PRIMARY.set_index('scenario').loc[FINAL_STRESS]; y=100*p.mean_difference.to_numpy(); lo=y-100*p.ci95_low.to_numpy(); hi=100*p.ci95_high.to_numpy()-y
fig,ax=plt.subplots(figsize=(8.8,5)); x=np.arange(len(FINAL_STRESS)); ax.errorbar(x,y,yerr=np.vstack([lo,hi]),fmt='o',capsize=4); ax.axhline(0,linewidth=1); ax.set_xticks(x); ax.set_xticklabels(FINAL_STRESS,rotation=20,ha='right'); ax.set_ylabel('DAPT×AL-Interaktion [pp Recall]'); ax.set_title('Independent Replication10 – Difference-in-Differences'); fig.tight_layout(); fig.savefig(FIG_ROOT/'FIG02_interaction.png',dpi=220); plt.close(fig)

if len(completed)!=10: raise RuntimeError('Completion only allowed for all 10 seeds')
ss=STRESS_STATS[STRESS_STATS.contrast=='INTERACTION_DAPT_AL_GAIN_MINUS_T0_AL_GAIN'].iloc[0]
if ss.mean_difference>0 and ss.ci95_low>0 and ss.exact_signflip_p_two_sided<.05: reading='REPLICATED_SUPPORT: positive DAPT×AL interaction replicated and statistically supported in stress mean.'
elif ss.mean_difference>0: reading='DESCRIPTIVE_ONLY: positive replication estimate, but interaction not statistically supported.'
else: reading='NO_REPLICATION_SUPPORT: independent replication does not show a positive DAPT×AL interaction in stress mean.'
completion={'status':'COMPLETE','run':'INDEPENDENT_REPLICATION10_DAPT_X_ACTIVE_LEARNING','replication_seeds':REPLICATION10,'performance_based_seed_exclusions':[],'posthoc_seed_additions':[],'fresh_dapt_per_seed':True,'primary_test':{'interaction':'(DAPT_U-DAPT_R)-(T0_U-T0_R)','metric':'recall','budget':.25,'target_fpr':.005,'scenarios':FINAL_STRESS,'stress_mean':{k:(float(ss[k]) if isinstance(ss[k],(int,float,np.integer,np.floating)) and pd.notna(ss[k]) else ss[k]) for k in ['n_seeds','mean_difference','ci95_low','ci95_high','exact_signflip_p_two_sided','cohen_dz','wins','ties','losses']}},'replication_reading':reading,'scientific_boundary':'Positive interaction supports the investigated DAPT + uncertainty-sampling combination under this protocol; it does not by itself establish a causal mechanism such as superior uncertainty calibration.'}
(ROOT/'REPLICATION10_COMPLETE.json').write_text(json.dumps(completion,indent=2,default=str),encoding='utf-8')
(ROOT/'README_REPLICATION10.md').write_text(f'Independent Replication10\nSeeds: {REPLICATION10}\nPrimary: (DAPT_U-DAPT_R)-(T0_U-T0_R), 25% labels, 0.5% target FPR.\nInterpretation: {reading}\nNo performance-based seed exclusions or post-hoc additions.',encoding='utf-8')
zip_path=Path('/kaggle/working/phreshphish_AL_SSL_INTERACTION_INDEPENDENT_REPLICATION10_ANALYSIS.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for pth in ROOT.rglob('*'):
        if pth.is_file() and DAPT_ROOT not in pth.parents: z.write(pth,arcname=str(pth.relative_to(ROOT)))
write_progress('COMPLETE',completed_seeds=REPLICATION10,analysis_zip=str(zip_path))
print(json.dumps(completion,indent=2,default=str)); print({'ANALYSIS_ZIP':str(zip_path)})

## Nach erfolgreichem Lauf sichern

- `phreshphish_AL_SSL_INTERACTION_INDEPENDENT_REPLICATION10_ANALYSIS.zip`
- `REPLICATION10_COMPLETE.json`
- `REPLICATION10_results_long.csv`
- `REPLICATION10_statistics.csv`
- `REPLICATION10_acquisition_audit.csv`
- `tables/`
- `figures/`
- `audit/PREDECLARED_SEED_PLAN.json`
- `audit/seed_*_selection.json`

Die **Independent Replication10** wird zuerst separat interpretiert. Das Zusammenführen mit den ursprünglichen zehn Seeds zu einer Pooled20-Präzisionsanalyse erfolgt erst danach als sekundäre Analyse.